In [22]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent,Runner,OpenAIChatCompletionsModel,function_tool
import os
from IPython.display import Markdown,display
import requests
from ddgs import DDGS

In [8]:
load_dotenv(override=True)

True

In [3]:
client=AsyncOpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
model=OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile",
    openai_client=client
)

In [28]:
@function_tool
def send_notification(message: str, title: str = "Agent Notification") -> str:
    """
    Sends a push notification to the user's devices via the Pushover API.
    
    Args:
        message: The body text of the push notification.
        title: The optional title of the notification.
    """
    # Retrieve credentials from your environment variables
    user_key = os.getenv("PUSHOVER_USER")
    api_token = os.getenv("PUSHOVER_TOKEN")
    if not user_key or not api_token:
        return "Error: Pushover credentials are not configured in environment variables."

    url = "https://api.pushover.net/1/messages"
    data = {
        "token": api_token,
        "user": user_key,
        "message": message,
        "title": title
    }
    
    try:
        response = requests.post(url, data=data)
        if response.status_code == 200:
            return "Notification sent successfully via Pushover."
        else:
            return f"Failed to send notification. Pushover API response: {response.text}"
    except Exception as e:
        return f"An error occurred while sending notification: {str(e)}"


In [21]:
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""

    api_key = os.getenv("OPENWEATHER_API_KEY")

    if not api_key:
        return "OPENWEATHER_API_KEY is not configured."

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": api_key,
        "units": "metric",
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()

        data = response.json()

        temperature = data["main"]["temp"]
        feels_like = data["main"]["feels_like"]
        humidity = data["main"]["humidity"]
        description = data["weather"][0]["description"]

        return (
            f"Weather in {city}:\n"
            f"Temperature: {temperature}°C\n"
            f"Feels like: {feels_like}°C\n"
            f"Condition: {description}\n"
            f"Humidity: {humidity}%"
        )

    except requests.exceptions.HTTPError:
        return f"Could not find weather data for {city}."

    except requests.exceptions.RequestException:
        return "Weather service is currently unavailable."

In [27]:
@function_tool
def search_tool(query: str) -> str:
    """Search the web using DuckDuckGo and return relevant results."""

    try:
        results = DDGS().text(
            query,
            max_results=5
        )

        if not results:
            return "No search results found."

        return "\n\n".join(
            f"Title: {result['title']}\n"
            f"URL: {result['href']}\n"
            f"Snippet: {result['body']}"
            for result in results
        )

    except Exception as e:
        return f"Search failed: {str(e)}"

In [52]:
research_agent=Agent(
    name="Research Agent",
    instructions="""
    You are a Research Agent with access to three tools:

- get_weather: Use only for weather-related queries.
- search_tool: Use only for web research and information retrieval.
- send_notification: Use only when the user explicitly requests a notification.

Rules:
1. Identify the user's intent before using any tool.
2. Call only the tool that directly matches the user's request.
3. Never call a tool when the query is unrelated to its purpose.
4. If the query is unrelated to all available tools, do not answer and do not call any tool.
5. For research queries, use search_tool and present the results as clear bullet points and concise paragraphs.
6. Do not invent information when a relevant tool is available.
7. Keep responses clear, concise, and well-structured.
    """,
    model=model,
    tools=[get_weather,search_tool,send_notification]
)

In [54]:
response=await Runner.run(
    research_agent,"send notification about islamabad weather condition"
)

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [55]:
display(Markdown(response.final_output))

You have been notified about the current weather in Islamabad. If you would like more updates or have other requests, please let me know.